# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malang43/flyrank-ml-internship-Malang43/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Lane: Refresh / Content Opportunity Scoring.
Unit of analysis: One row represents one content page for one pseudonymized client after daily observations are aggregated over a defined time window.
Tables: I will use fact_content_daily_performance for search and performance signals and dim_content when content-level context is needed.
Time window: Time window: Time window: March 1–31, 2026 is the feature window. April 1–30, 2026 is used as a provisional future outcome window. The two windows do not overlap.. Features will be based on information available before the outcome window so that future information is not used at prediction time.
Goal: Rank content pages by review priority so that a human reviewer can decide which pages deserve attention first.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features — maximum five:

GSC impressions
GSC clicks
CTR
Average search position
Sessions / engagement signal where measurement is available

future_decline_proxy = 1 when a page's April GSC impressions are at least 20% lower than its March impressions; otherwise 0. This is a provisional future-performance proxy, not proof that a page requires refreshing.

Context: client_hash_id, content_hash_id, report date, and measurement-availability flags are used for joining, grouping, and validation, not as predictive signals themselves.

Excluded: Any label-derived field, information from the future outcome window, product decision score, private URL/query/client information, or measurement unavailable at the decision time. These are excluded because they can cause leakage, circular results, or privacy problems.

In [19]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "avg_position",
    "sessions"
]

print("Planned features:")
for x in features:
    print("-", x)

Planned features:
- gsc_impressions
- gsc_clicks
- ctr
- avg_position
- sessions


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
!pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [token])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to March 2026 warehouse data.")

Connected to March 2026 warehouse data.


In [21]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        client_hash_id || '|' ||
        content_hash_id
    ) AS unique_client_content_day
FROM {MAR}
""").df()

q1

,rows,unique_client_content_day
0,9841378,9841378


The matching counts verify that the raw March fact table is at client × content × day grain.

In [22]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages
FROM {MAR}
""").df()

q2

,row_count,min_date,max_date,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


In [23]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS usable_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS usable_pct
FROM {MAR}
""").df()

q3

,total_rows,usable_rows,usable_pct
0,9841378,364347,3.7


Availability was checked explicitly using IS TRUE. Rows without measurement availability should not automatically be interpreted as genuine zero performance.

Impressions: Knowable at the decision moment because they were observed during the completed March feature window.

Clicks: Knowable because they are search clicks already observed before the April outcome period.

CTR: Knowable because it is calculated only from March clicks and impressions.

Average position: Knowable because it summarizes search position measured during March.

Sessions: Knowable when GA4 tracking is available because the sessions occurred during March, before the April target window.

In [24]:
con.sql(f"""
SELECT *
FROM {MAR}
LIMIT 1
""").df().columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [25]:
# March = feature window
march_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_31d,
    SUM(gsc_clicks) AS clicks_31d,

    SUM(gsc_clicks) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS ctr_31d,

    SUM(gsc_sum_position) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS avg_position_31d,

    COALESCE(SUM(ga4_sessions), 0) AS sessions_31d

FROM {MAR}

WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) > 0
""").df()

print("March feature rows:", len(march_features))

display(march_features.head(10))

March feature rows: 63856


,client_hash_id,content_hash_id,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,sessions_31d
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,0.003891,9.879377,42.0
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,0.005556,8.455556,7.0
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,0.010124,4.559801,168.0
3,client_9958f0a7ae1df715,content_278030b007943b07,319.0,7.0,0.021944,5.912226,16.0
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,6106.0,39.0,0.006387,7.069767,40.0
5,client_9958f0a7ae1df715,content_347fbafb77d3ae37,396.0,4.0,0.010101,21.734848,38.0
6,client_9958f0a7ae1df715,content_15bd72d24e0a0b08,428.0,2.0,0.004673,12.462617,15.0
7,client_9958f0a7ae1df715,content_6f4cc70af7be346e,133.0,5.0,0.037594,9.473684,11.0
8,client_9958f0a7ae1df715,content_c5c8a9160d8b64ae,502.0,4.0,0.007968,21.890438,26.0
9,client_9958f0a7ae1df715,content_85e499a6915dc41e,17.0,0.0,0.000000,7.588235,6.0


In [26]:
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

april_target = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions

FROM {APR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("April target rows:", len(april_target))

April target rows: 194760


In [27]:
ml_df = march_features.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

ml_df["future_decline_proxy"] = (
    ml_df["april_impressions"]
    < 0.80 * ml_df["impressions_31d"]
).astype(int)

print("Rows with March features + April outcome:", len(ml_df))

print("\nTarget distribution:")
print(ml_df["future_decline_proxy"].value_counts())

print("\nTarget percentages:")
print(
    ml_df["future_decline_proxy"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(
    ml_df[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_31d",
            "clicks_31d",
            "ctr_31d",
            "avg_position_31d",
            "sessions_31d",
            "future_decline_proxy"
        ]
    ].head(10)
)

Rows with March features + April outcome: 62681

Target distribution:
future_decline_proxy
0    60225
1     2456
Name: count, dtype: int64

Target percentages:
future_decline_proxy
0    96.08
1     3.92
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,sessions_31d,future_decline_proxy
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,0.003891,9.879377,42.0,1
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,0.005556,8.455556,7.0,1
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,0.010124,4.559801,168.0,1
3,client_9958f0a7ae1df715,content_278030b007943b07,319.0,7.0,0.021944,5.912226,16.0,1
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,6106.0,39.0,0.006387,7.069767,40.0,1
5,client_9958f0a7ae1df715,content_347fbafb77d3ae37,396.0,4.0,0.010101,21.734848,38.0,1
6,client_9958f0a7ae1df715,content_15bd72d24e0a0b08,428.0,2.0,0.004673,12.462617,15.0,0
7,client_9958f0a7ae1df715,content_6f4cc70af7be346e,133.0,5.0,0.037594,9.473684,11.0,0
8,client_9958f0a7ae1df715,content_c5c8a9160d8b64ae,502.0,4.0,0.007968,21.890438,26.0,1
9,client_9958f0a7ae1df715,content_85e499a6915dc41e,17.0,0.0,0.000000,7.588235,6.0,1


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

feature_cols = [
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "sessions_31d"
]

X = ml_df[feature_cols].fillna(0)
y = ml_df["future_decline_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, honest_prob)

print(f"Honest ROC AUC: {honest_auc:.3f}")

Honest ROC AUC: 0.720


In [29]:
ml_df["label_copy_leak"] = ml_df["future_decline_proxy"]

leaky_features = feature_cols + ["label_copy_leak"]

X_leaky = ml_df[leaky_features].fillna(0)
y = ml_df["future_decline_proxy"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_prob = leaky_model.predict_proba(X_test)[:, 1]

leaky_auc = roc_auc_score(y_test, leaky_prob)

print(f"Honest ROC AUC: {honest_auc:.3f}")
print(f"ROC AUC WITH target leakage: {leaky_auc:.3f}")

Honest ROC AUC: 0.720
ROC AUC WITH target leakage: 1.000


In [30]:
ml_df.drop(columns=["label_copy_leak"], inplace=True)

print("Leakage feature removed.")
print("Final modeling features:")
print(feature_cols)

print(f"\nScore retained for this exercise: {honest_auc:.3f}")

Leakage feature removed.
Final modeling features:
['impressions_31d', 'clicks_31d', 'ctr_31d', 'avg_position_31d', 'sessions_31d']

Score retained for this exercise: 0.720


Leakage lesson: When I deliberately copied information from the target into a feature, the score became unrealistically high because the model was effectively given the answer. This is target leakage, not genuine predictive performance. I removed the leaked feature and retained the honest score using only information available during the March decision window.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: The warehouse has unbalanced client history and not every metric is measured on every row. Some early rows may contain search data while analytics tracking was not yet available. Therefore, missing analytics measurements cannot automatically be interpreted as zero traffic or zero engagement. In addition, this observational dataset can support directional and decision-support conclusions, but it cannot prove that refreshing a page causes improved search performance.
This exercise also uses only one March-to-April transition, so the result does not establish generalization across other months or seasonal conditions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.